# 第9回　仮説検定(2)：分散分析（ANOVA）
## ―― 3つ以上を比べるとき、t検定を繰り返してはいけない

統計学Ⅰ（B）　／　北星学園大学

注目は ――

> 検定を**繰り返す**と、偽陽性（まぐれの「有意」）が**勝手に増えていく。**

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
from itertools import combinations

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## 1. なぜ「繰り返し」がダメなのか

1回のt検定は「本当は差がないのに、まぐれで有意」と誤る確率を **5%（α）** 許している。

では検定を**何回も**やったら？　毎回5%の落とし穴があるのだから、回数が増えるほど「どれか1つはまぐれで有意」になりやすい。これを確かめる。

---
## 2. 多重比較の罠（シミュレーション）

**本当はすべて同じ母集団（＝差がまったくない）** の k 個の群を作り、全ペアで t検定する。「最低1つでも有意（p<0.05）」になってしまう割合を数える。差はないのだから、本来5%であってほしい。

In [ ]:
rng = np.random.default_rng(2026)
ks = [2, 3, 5, 10]
偽陽性率 = []
for k in ks:
    fp = 0
    for _ in range(3000):
        群 = [rng.normal(50, 10, 30) for _ in range(k)]   # 全部おなじ＝差なし
        もし1つでも有意 = any(stats.ttest_ind(群[i], 群[j])[1] < 0.05
                          for i, j in combinations(range(k), 2))
        fp += もし1つでも有意
    率 = fp/3000*100
    偽陽性率.append(率)
    print(f"群数 k={k:2d}（{k*(k-1)//2:2d}ペア）: 最低1つ有意になった割合 = {率:3.0f}%（本来は5%のはず）")

In [ ]:
plt.figure(figsize=(6.5,3.6))
bars = plt.bar([str(k) for k in ks], 偽陽性率, color="#e8503a")
plt.axhline(5, color="gray", ls="--", label="本来あるべき 5%")
for b,r in zip(bars,偽陽性率): plt.text(b.get_x()+b.get_width()/2, r+1.5, f"{r:.0f}%", ha="center")
plt.xlabel("群の数 k"); plt.ylabel("まぐれで有意になった割合")
plt.title("差がないのに、群を増やすほど偽陽性が増える"); plt.legend(); plt.show()

**差はまったくないのに**、群が増える（＝ペア＝検定回数が増える）ほど「最低1つ有意」が増える。10群なら6割以上がまぐれの「発見」だ。

だから3群以上を「全ペアt検定」してはいけない。**検定の回数を増やすほど、偽の発見が紛れ込む。**

---
## 3. ANOVA ―― まず1回で「どこかに差があるか」を判定

分散分析（ANOVA）は、3群以上を**一度の検定**で「**どこかに差があるか**」だけを判定する。検定を繰り返さないので偽陽性が増えない。

**3つの科の体重**で実行する（`scipy.stats.f_oneway`）。第3回でやったとおり、体重は対数の目盛りで扱う。

In [ ]:
三科 = ["オマキザル科", "クモザル科", "コビトキツネザル科"]
s = df[df["科"].isin(三科)].dropna(subset=["体重g"])

for f, g in s.groupby("科"):
    print(f"{f:<12} n={len(g):3d}  中央値 {g['体重g'].median():>8,.0f} g")

群 = [np.log10(g["体重g"].values) for _, g in s.groupby("科")]
F, p = stats.f_oneway(*群)
print(f"\nANOVA: F値 = {F:.2f}, p値 = {p:.3g}")
if p < 0.05:
    print("→ p<0.05。どこかの科の間に差がありそう（次に事後検定でどのペアか調べる）")

**F = 151.89、p は 3×10⁻²⁵。** 圧倒的に有意である。オマキザル科492g・クモザル科6,652g・コビトキツネザル科69g ―― 中央値が桁で違うのだから当然だ。

### F値の気持ち

ANOVAは2種類のばらつきを比べる。

$$ F = \frac{群間のばらつき（グループの平均どうしの違い）}{群内のばらつき（同じグループ内の個体差）} $$

- F が大きい ＝ グループ間の差が、個体差に比べて大きい → 「差がある」
- F が1前後 ＝ グループの違いは個体差の範囲 → 「差があるとは言えない」

F=151.89 は「グループ間の差が、群内のばらつきの150倍」という意味である。

---
## 3b. 対照実験 ―― F が1前後だと、どう見えるか

同じ科の枠組みで、**差がなさそうな量**を調べてみる。「科によって、生息地の降水量に差はあるか？」

In [ ]:
三科2 = ["オナガザル科", "サキ科", "クモザル科"]
s2 = df[df["科"].isin(三科2)].dropna(subset=["月降水量mm"])

for f, g in s2.groupby("科"):
    print(f"{f:<10} n={len(g):3d}  平均 {g['月降水量mm'].mean():>6.1f} mm")

群2 = [g["月降水量mm"].values for _, g in s2.groupby("科")]
F2, p2 = stats.f_oneway(*群2)
print(f"\nANOVA: F値 = {F2:.2f}, p値 = {p2:.3f}")
if p2 >= 0.05:
    print("→ p≥0.05。平均は少し違って見えるが、群内のばらつきを考えると")
    print("  『科によって生息地の降水量に差がある』とは言えない。")

**F = 0.08、p = 0.92。** 平均値の見かけの違いは、群内のばらつきに完全に埋もれている。

> **同じデータセット・同じ手法でも、変数によって結果はまったく違う。**
> 「有意でなかった」も立派な結果である。**差がないことを確かめた**のだから。

注意すべきは、これは「差がないことの証明」ではないこと。**「この人数・このばらつきでは、差を検出できなかった」**が正確な言い方である。

---
## 4. ANOVAが有意だったら ―― 事後検定と補正

ANOVAが教えるのは「**どこかに**差がある」まで。**どのペアか**は別途調べる（事後検定）。

その際も検定を繰り返すので、**有意水準を厳しくする補正**をかける。

- **ボンフェローニ補正**：α を検定回数で割る。3群なら3ペアなので **α = 0.05 ÷ 3 = 0.0167** を各検定の基準にする。

§3で有意だった3つの科で、実際にやってみる。

In [ ]:
名前 = [f for f, _ in s.groupby("科")]
ペア数 = len(list(combinations(range(3), 2)))
α補正 = 0.05 / ペア数

print(f"ペア数 = {ペア数}　→　補正後の基準 α = 0.05 / {ペア数} = {α補正:.4f}\n")
for (i, a), (j, b) in combinations(enumerate(群), 2):
    t, pp = stats.ttest_ind(a, b, equal_var=False)
    判定 = "有意" if pp < α補正 else "有意でない"
    print(f"  {名前[i]:<12} vs {名前[j]:<12}  p = {pp:.3g}   → {判定}")
print("\n→ 3ペアすべてが補正後も有意。どの科どうしも体重が違う。")

今回は補正しても3ペアすべてが有意だった。**中央値が桁で違うのだから、当然である。**

だが忘れてはいけない。**補正をかけなければ、まぐれで有意になるペアが紛れ込む。**§2で見たとおり、5群（10ペア）なら4割、10群（45ペア）なら6割以上がまぐれで「発見」される。

> ❌ よくある誤り：「ANOVAが有意 ＝ すべての群が互いに違う」。
> 正しくは「**どこかに**差がある」だけ。どのペアかは事後検定で確かめる。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 多重比較の罠 | 検定を繰り返すほど、まぐれの有意（偽陽性）が増える（10群で6割超） |
| ANOVA | 3群以上を1回で「どこかに差があるか」判定。偽陽性が増えない |
| F値 | 群間のばらつき ÷ 群内のばらつき。大きいほど差あり |
| 事後検定＋補正 | どのペアかは別途。ボンフェローニ補正で α を回数で割る |
| 「有意でない」も結果 | ただし「差がない証明」ではなく「検出できなかった」 |
| ❌ 誤り | 全ペアt検定でOK／ANOVA有意＝全群が違う |

> **群が3つ以上なら、t検定を繰り返さない。まずANOVAで一括判定。**
> 「有意」は『どこかに差』であって『全部違う』ではない。

> **なお、今日のF=151.89 は「差が大きい」ことを意味するが、「その差が重要だ」とは言っていない。**
> 検定は「差があるか」しか答えない。**どれくらいの差か**は別の問い（効果量）である。
> これは統計学Ⅱ第9回で扱う。

**課題（Moodle）**：あるシナリオで「t検定の繰り返し」と「ANOVA」のどちらが適切か、理由とともに判断する。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。